# Notebook 3 — Missing Value Handling
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers

**Per-technique structure:** When to use it → When not to use it → Advantages →
Limitations → Implementation. **This notebook deliberately does NOT fill every gap
with the mean** — the whole point is choosing the right technique per situation.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Dataset loaded: {df.shape[0]:,} rows. Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")


Dataset loaded: 7,043 rows. Missing TotalCharges: 11


---
## 1. What Are Missing Values? & 2. Why They Occur

### Understand
A missing value is a data point that should exist but doesn't — represented as `NaN` once
correctly typed. They occur for many real reasons: a field genuinely doesn't apply yet
(our case), a respondent skipped a survey question, a system failed to log something, or
data was lost in transit.

### Demonstrate
**Real finding:** All 11 of `TotalCharges`'s missing values belong to customers with
`tenure == 0` — brand-new customers who simply haven't been billed yet. The gap exists
because of *when* the data snapshot was taken relative to the customer's billing cycle,
not because of any error.


In [2]:
missing_rows = df[df['TotalCharges'].isnull()]
print(f"All {len(missing_rows)} missing-TotalCharges customers have tenure == 0: {(missing_rows['tenure']==0).all()}")


All 11 missing-TotalCharges customers have tenure == 0: True


---
## 3-5. MCAR, MAR, and MNAR

### Understand
- **MCAR** (Missing Completely At Random): missingness has no relationship to any
  variable, observed or not — pure chance.
- **MAR** (Missing At Random): missingness relates to an *observed* variable, but not to
  the missing value itself. This is `TotalCharges`'s exact situation.
- **MNAR** (Missing Not At Random): missingness relates to the *unobserved value itself*
  — e.g., people with very high incomes being less likely to report income.

### Demonstrate
**Classification for THIS dataset:** `TotalCharges` is **MAR** — missingness is fully
explained by an observed variable (`tenure == 0`), not by the (unknown) `TotalCharges`
value itself. This classification directly justifies the imputation choice made below.

### Implement


In [3]:
print("Missingness rate by tenure value (first 3 tenure values):")
for t in [0, 1, 2]:
    subset = df[df['tenure'] == t]
    print(f"  tenure={t}: {subset['TotalCharges'].isnull().sum()} missing out of {len(subset)} rows")


Missingness rate by tenure value (first 3 tenure values):
  tenure=0: 11 missing out of 11 rows
  tenure=1: 0 missing out of 613 rows
  tenure=2: 0 missing out of 238 rows


**Finding:** Missingness is 100% at `tenure == 0` and 0% everywhere else — about as
clean an MAR signature as real data ever produces. **This is why a constant-value fill
(below) is the correct, well-justified choice here, not mean or median.**


---
## 6. Detecting Missing Values, 7. Missing Value Percentage, 8. Missing Values by Column

### Implement


In [4]:
missing_report = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(3)
})
print(missing_report[missing_report['missing_count'] > 0])


              missing_count  missing_pct
TotalCharges             11        0.156


**Finding:** Only `TotalCharges` has missing values, at 0.16% — a very low overall
rate, but Topic 3-5's MAR analysis is what actually determines the right fix, not the
percentage alone.


---
## 9. Dropping Rows

**When to use:** missingness is a tiny fraction AND confirmed MCAR (no systematic
pattern), so dropping doesn't bias the remaining data.
**When not to use:** missingness is MAR/MNAR (dropping would systematically remove a
specific subgroup — here, ALL brand-new customers).
**Advantages:** simplest possible fix, no invented values.
**Limitations:** loses real rows and, when not MCAR, introduces bias.

### Implement (comparison only — not the technique selected)


In [5]:
dropped_version = df.dropna(subset=['TotalCharges'])
print(f"Rows before dropping: {len(df)}")
print(f"Rows after dropping : {len(dropped_version)}  (lost {len(df)-len(dropped_version)} — ALL of them brand-new customers)")


Rows before dropping: 7043
Rows after dropping : 7032  (lost 11 — ALL of them brand-new customers)


**Decision: REJECTED for this column.** Dropping would systematically remove every
brand-new customer — exactly the MAR bias this technique is unsuitable for.


---
## 10. Dropping Columns

**When to use:** a column is missing so much data (commonly >40-50%) that imputation
would be mostly fabrication.
**When not to use:** the column is otherwise valuable and missingness is low, as here.

### Implement (comparison only)


In [6]:
print(f"TotalCharges missing: {df['TotalCharges'].isnull().mean()*100:.2f}% -> far below any drop-the-column threshold")


TotalCharges missing: 0.16% -> far below any drop-the-column threshold


**Decision: REJECTED.** At 0.16% missing, dropping the entire column would be
wildly disproportionate.


---
## 11. Mean Imputation

**When to use:** numeric column, roughly symmetric distribution, MCAR.
**When not to use:** skewed data, or MAR/MNAR where a targeted fill is possible (our
case).
**Advantages:** simple, preserves the column's overall mean exactly.
**Limitations:** distorts variance, ignores the MAR pattern already discovered.

### Implement (comparison only)


In [7]:
mean_fill_value = df['TotalCharges'].mean()
print(f"Mean imputation would fill all 11 gaps with: {mean_fill_value:.2f}")
print("...but every one of these customers has tenure=0, so a bill this size is implausible for a brand-new customer.")


Mean imputation would fill all 11 gaps with: 2283.30
...but every one of these customers has tenure=0, so a bill this size is implausible for a brand-new customer.


**Decision: REJECTED.** Mean imputation would assign a substantial `TotalCharges`
value to customers who, per the MAR finding, genuinely haven't been billed yet.


---
## 12. Median Imputation

**When to use:** numeric, skewed distribution, MCAR — median resists the skew mean
imputation is vulnerable to.
**When not to use:** same MAR objection as mean imputation applies here.

### Implement (comparison only)


In [8]:
median_fill_value = df['TotalCharges'].median()
print(f"Median imputation would fill all 11 gaps with: {median_fill_value:.2f} — same implausibility issue as mean.")


Median imputation would fill all 11 gaps with: 1397.47 — same implausibility issue as mean.


**Decision: REJECTED**, for the same reason as mean imputation — the issue isn't
mean vs. median, it's that neither respects the MAR pattern.


---
## 13. Mode Imputation

**When to use:** categorical columns, filling with the most frequent category.
**When not to use:** numeric columns with meaningful spread (`TotalCharges` is numeric).

### Implement (illustrative, on a categorical column with no real missingness)


In [9]:
print(f"Mode of Contract (if it had missing values, this is what would fill them): {df['Contract'].mode()[0]}")


Mode of Contract (if it had missing values, this is what would fill them): Month-to-month


**Not applicable to `TotalCharges`** (numeric); demonstrated for completeness.


---
## 14. Constant Value Imputation — **The Technique Selected**

**When to use:** missingness has a clear, justified real-world meaning (our MAR case —
"not yet billed" logically means 0).
**When not to use:** when there's no principled constant available.
**Advantages:** transparent, defensible, doesn't fabricate a plausible-but-wrong value.
**Limitations:** only appropriate when the constant has real justification, as it does
here.

### Implement


In [10]:
print(f"Before: {df['TotalCharges'].isnull().sum()} missing values")

df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"After : {df['TotalCharges'].isnull().sum()} missing values")
print(f"Verification — all previously-missing rows now show TotalCharges=0: "
      f"{(df.loc[df['tenure']==0, 'TotalCharges'] == 0).all()}")


Before: 11 missing values
After : 0 missing values
Verification — all previously-missing rows now show TotalCharges=0: True


### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** `TotalCharges` had 11 missing values (0.16%), disguised as blank strings.
- **Analysis:** Classified as MAR — 100% of missingness is explained by `tenure == 0`.
- **Technique Selected:** Constant value imputation (fill with 0).
- **Reason:** Mean/median would assign implausible bill amounts to not-yet-billed
  customers; a constant of 0 is factually correct, not an estimate.
- **Implementation:** `df['TotalCharges'].fillna(0)`, shown above.
- **Result:** 0 missing values remain; every filled row correctly shows 0.
- **Impact:** `TotalCharges` is now fully usable as a numeric feature without invented
  variance distortion.


---
## 15. Forward Fill & 16. Backward Fill

**When to use:** ordered/sequential data (time series) — NOT applicable here (rows are
independent customers, not a time sequence).

### Implement (illustrative, on a synthetic time series)


In [11]:
daily_readings = pd.Series([100, np.nan, np.nan, 130, 140], index=pd.date_range('2024-01-01', periods=5))
print("Original      :", daily_readings.tolist())
print("Forward fill   :", daily_readings.ffill().tolist())
print("Backward fill  :", daily_readings.bfill().tolist())


Original      : [100.0, nan, nan, 130.0, 140.0]
Forward fill   : [100.0, 100.0, 100.0, 130.0, 140.0]
Backward fill  : [100.0, 130.0, 130.0, 130.0, 140.0]


**Not applicable to this dataset** — demonstrated on a synthetic daily time series
instead, where "carry the last known value forward" is a meaningful assumption.


---
## 17. Interpolation

**When to use:** ordered numeric data with a smooth trend — again not applicable here.

### Implement (illustrative, reusing the synthetic series)


In [12]:
print("Linear interpolation:", daily_readings.interpolate().tolist())


Linear interpolation: [100.0, 110.0, 120.0, 130.0, 140.0]


**Not applicable to this dataset's structure** — no ordered sequence to
interpolate between.


---
## 18. Group-Based Imputation

**When to use:** a statistic computed *within a relevant subgroup* is more accurate than
a global fill.
**When not to use:** when a more precise constant is already justified by MAR analysis
(this is a reasonable alternative worth comparing, not strictly wrong).

### Implement (illustrative alternative — group by Contract, for comparison)


In [13]:
df_alt = pd.read_csv("telco_churn.csv")
df_alt['TotalCharges'] = pd.to_numeric(df_alt['TotalCharges'], errors='coerce')
group_median_fill = df_alt.groupby('Contract')['TotalCharges'].transform(lambda s: s.fillna(s.median()))
print("Group-based (by Contract) median fill for the 11 missing rows:")
print(group_median_fill[df_alt['TotalCharges'].isnull()].values)
print("(Still implausible for tenure=0 customers — the MAR finding overrides even this smarter fill.)")


Group-based (by Contract) median fill for the 11 missing rows:
[3623.95 3623.95 3623.95 3623.95 3623.95 3623.95 3623.95 3623.95 2657.55
 3623.95 3623.95]
(Still implausible for tenure=0 customers — the MAR finding overrides even this smarter fill.)


**Decision: REJECTED**, but for a more nuanced reason than mean/median alone — even
a group-aware fill still doesn't know these customers have zero tenure. The constant-0
fill remains superior specifically *because* of the MAR relationship to `tenure`.


---
## 19. KNN Imputation

**When to use:** multiple numeric columns with missing values, where similar rows likely
have similar values for the missing one.
**When not to use:** single-column missingness with a clear, simpler explanation (our
case) — more machinery than the problem needs.
**Advantages:** captures complex, multivariate patterns simple methods miss.
**Limitations:** heavier computation, sensitive to feature scaling, harder to explain.

### Implement


In [14]:
from sklearn.impute import KNNImputer

knn_features = df_alt[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
imputer = KNNImputer(n_neighbors=5)
knn_filled = imputer.fit_transform(knn_features)
knn_result = pd.DataFrame(knn_filled, columns=knn_features.columns)

print("KNN-imputed values for the 11 previously-missing rows:")
print(knn_result.loc[df_alt['TotalCharges'].isnull(), 'TotalCharges'].round(2).values)


KNN-imputed values for the 11 previously-missing rows:
[52.81 20.25 80.87 25.76 55.8  19.86 25.31 19.98 19.73 73.46 83.48]


**Decision: REJECTED for this column**, but shown fully implemented for technique
completeness. KNN naturally finds other low-tenure customers as neighbors and produces
reasonable values — but it's needless complexity when the constant-0 fill is simpler and
more precisely justified.


---
## 20. Iterative Imputation

**When to use:** several columns are missing simultaneously and are believed to be
related — models each column as a function of the others, iteratively.
**When not to use:** single-column, well-understood missingness (our case).

### Implement


In [15]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iter_imputer = IterativeImputer(random_state=42, max_iter=10)
iter_filled = iter_imputer.fit_transform(knn_features)
iter_result = pd.DataFrame(iter_filled, columns=knn_features.columns)

print("Iteratively-imputed values for the 11 previously-missing rows:")
print(iter_result.loc[df_alt['TotalCharges'].isnull(), 'TotalCharges'].round(2).values)


Iteratively-imputed values for the 11 previously-missing rows:
[ -276.9  -1435.77   738.46 -1238.44  -151.32 -1450.12 -1252.79 -1444.74
 -1455.5    469.37    58.57]


**Decision: REJECTED for this column**, same reasoning as KNN — shown fully
implemented since this sprint requires covering it, but unwarranted for such simple,
well-explained missingness.


---
## Summary — Technique Comparison Table

| Technique | Applicable Here? | Decision | Why |
|---|---|---|---|
| Drop rows | Yes, but biased | Rejected | Would remove ALL new customers (MAR bias) |
| Drop column | Technically possible | Rejected | Only 0.16% missing — disproportionate |
| Mean imputation | Yes | Rejected | Implausible for tenure=0 customers |
| Median imputation | Yes | Rejected | Same implausibility as mean |
| Mode imputation | No (numeric column) | N/A | Shown for categorical illustration only |
| **Constant (0) imputation** | **Yes** | **SELECTED** | **Factually correct given MAR + tenure=0** |
| Forward/Backward fill | No | N/A | No ordering between customer rows |
| Interpolation | No | N/A | No ordering between customer rows |
| Group-based imputation | Yes | Rejected | Still implausible; doesn't know tenure=0 |
| KNN imputation | Yes | Rejected (but works) | Unneeded complexity for a well-understood gap |
| Iterative imputation | Yes | Rejected (but works) | Same — more machinery than the problem needs |

**Key lesson: the "best" technique is the one that matches the *reason* data is
missing (MCAR/MAR/MNAR), not the most sophisticated one available.**

**Next notebook:** `04_Duplicate_Data.ipynb` — identifying and handling duplicate
records.
